# Chemical Structure Visualization with Atom Numbers

This notebook provides functions to display chemical structures with atom numbers for all non-hydrogen atoms using RDKit.

In [ ]:
# Import required libraries
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import display
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
import pandas as pd

## Function: Display Molecule with Atom Numbers

This function takes a SMILES string or RDKit molecule object and displays it with atom numbers for all non-hydrogen atoms.

In [ ]:
def display_molecule_with_atom_numbers(mol_input, size=(400, 400), highlight_atoms=None):
    """
    Display a chemical structure with atom numbers for all non-hydrogen atoms.
    
    Parameters:
    -----------
    mol_input : str or rdkit.Chem.Mol
        Either a SMILES string or an RDKit molecule object
    size : tuple, optional
        Size of the output image (width, height). Default is (400, 400)
    highlight_atoms : list, optional
        List of atom indices to highlight. Default is None
    
    Returns:
    --------
    PIL.Image
        The molecule image with atom numbers
    
    Examples:
    ---------
    >>> # Display from SMILES string
    >>> display_molecule_with_atom_numbers('CCO')
    
    >>> # Display with custom size
    >>> display_molecule_with_atom_numbers('c1ccccc1', size=(600, 600))
    
    >>> # Display with highlighted atoms
    >>> display_molecule_with_atom_numbers('CCO', highlight_atoms=[0, 2])
    """
    # Convert SMILES to molecule if needed
    if isinstance(mol_input, str):
        mol = Chem.MolFromSmiles(mol_input)
        if mol is None:
            raise ValueError(f"Invalid SMILES string: {mol_input}")
    else:
        mol = mol_input
    
    # Create a copy to avoid modifying the original molecule
    mol_copy = Chem.Mol(mol)
    
    # Add atom numbers to all non-hydrogen atoms
    for atom in mol_copy.GetAtoms():
        if atom.GetSymbol() != 'H':
            atom.SetProp('atomLabel', str(atom.GetIdx()))
    
    # Generate 2D coordinates if not present
    if mol_copy.GetNumConformers() == 0:
        AllChem.Compute2DCoords(mol_copy)
    # Draw the molecule
    drawer = Draw.MolDraw2DCairo(size[0], size[1])
    
    # Set drawing options
    drawer.drawOptions().addAtomIndices = False  # We're adding custom labels
    
    # Draw with optional highlighting
    if highlight_atoms:
        drawer.DrawMolecule(mol_copy, highlightAtoms=highlight_atoms)
    else:
        drawer.DrawMolecule(mol_copy)
    
    drawer.FinishDrawing()
    
    # Convert to PIL Image
    img_bytes = drawer.GetDrawingText()
    img = Image.open(BytesIO(img_bytes))
    
    return img

## Example Usage

Let's test the function with various molecules:

In [ ]:
# Example 1: Simple molecule - Ethanol
print("Example 1: Ethanol (CCO)")
img1 = display_molecule_with_atom_numbers('CCO')
display(img1)

In [ ]:
# Example 2: Benzene
print("Example 2: Benzene (c1ccccc1)")
img2 = display_molecule_with_atom_numbers('c1ccccc1')
display(img2)

In [ ]:
# Example 3: Aspirin with larger size
print("Example 3: Aspirin (CC(=O)Oc1ccccc1C(=O)O)")
img3 = display_molecule_with_atom_numbers('CC(=O)Oc1ccccc1C(=O)O', size=(600, 600))
display(img3)

In [ ]:
# Example 4: Caffeine with highlighted atoms
print("Example 4: Caffeine with highlighted nitrogen atoms")
caffeine = Chem.MolFromSmiles('CN1C=NC2=C1C(=O)N(C(=O)N2C)C')

# Find all nitrogen atoms
nitrogen_atoms = [atom.GetIdx() for atom in caffeine.GetAtoms() if atom.GetSymbol() == 'N']
print(f"Nitrogen atoms at indices: {nitrogen_atoms}")

img4 = display_molecule_with_atom_numbers(caffeine, size=(600, 600), highlight_atoms=nitrogen_atoms)
display(img4)

## Working with SMARTS Patterns

You can also use this function to visualize SMARTS patterns by converting them to molecules:

In [ ]:
# Example: Visualize a SMARTS pattern
# Note: SMARTS patterns need to be converted to molecules for visualization
smarts_pattern = '[#6]-[#8]'  # Carbon-Oxygen bond
smarts_mol = Chem.MolFromSmarts(smarts_pattern)

if smarts_mol:
    print(f"SMARTS pattern: {smarts_pattern}")
    img5 = display_molecule_with_atom_numbers(smarts_mol)
    display(img5)
else:
    print("Invalid SMARTS pattern")

## Additional Helper Function: Get Atom Information

This function provides detailed information about each atom in a molecule:

In [ ]:
def get_atom_info(mol_input):
    """
    Get detailed information about all non-hydrogen atoms in a molecule.
    
    Parameters:
    -----------
    mol_input : str or rdkit.Chem.Mol
        Either a SMILES string or an RDKit molecule object
    
    Returns:
    --------
    list of dict
        List of dictionaries containing atom information
    """
    # Convert SMILES to molecule if needed
    if isinstance(mol_input, str):
        mol = Chem.MolFromSmiles(mol_input)
        if mol is None:
            raise ValueError(f"Invalid SMILES string: {mol_input}")
    else:
        mol = mol_input
    
    atom_info = []
    for atom in mol.GetAtoms():
        if atom.GetSymbol() != 'H':
            info = {
                'index': atom.GetIdx(),
                'symbol': atom.GetSymbol(),
                'atomic_num': atom.GetAtomicNum(),
                'degree': atom.GetDegree(),
                'valence': atom.GetTotalValence(),
                'formal_charge': atom.GetFormalCharge(),
                'hybridization': str(atom.GetHybridization()),
                'aromatic': atom.GetIsAromatic(),
                'in_ring': atom.IsInRing()
            }
            atom_info.append(info)
    
    return atom_info

In [ ]:
# Example: Get atom information for benzene
import pandas as pd

benzene_info = get_atom_info('c1ccccc1')
df = pd.DataFrame(benzene_info)
print("Atom information for benzene:")
display(df)